# Diagrammi per il documento OmniRetail AI Governance

Genera i sette PNG referenziati dal markdown principale. Eseguire dall'alto in basso. Ogni cella produce un file `0N_*.png` nella stessa directory.

Stile tech unificato: palette light-tech con accenti vivaci, tipografia monospace per chip e badge, card con accent bar, sfondo a dot-grid. Dimensioni coerenti, esportazione 170 DPI per riuso in Google Docs.


In [1]:
"""
Tech-style diagram generator for the OmniRetail AI Governance capstone.
Run this script from inside the diagrams/ directory to regenerate all 7 PNGs.
This file is a staging area; the canonical source is the notebook 00_generate_diagrams.ipynb.
"""

from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Polygon, Circle
from matplotlib.lines import Line2D
import numpy as np


# === Palette and shared style ============================================

BG          = '#F8FAFC'
SURFACE     = '#FFFFFF'
BORDER      = '#E2E8F0'
BORDER_SOFT = '#F1F5F9'
INK         = '#0F172A'
INK_SOFT    = '#475569'
INK_MUTED   = '#94A3B8'

ACCENT = {
    'blue':    '#2563EB',
    'cyan':    '#06B6D4',
    'emerald': '#10B981',
    'amber':   '#F59E0B',
    'rose':    '#F43F5E',
    'violet':  '#8B5CF6',
    'slate':   '#64748B',
}

TINT = {  # 12 percent washes
    'blue':    '#DBEAFE',
    'cyan':    '#CFFAFE',
    'emerald': '#D1FAE5',
    'amber':   '#FEF3C7',
    'rose':    '#FFE4E6',
    'violet':  '#EDE9FE',
    'slate':   '#E2E8F0',
}

DPI = 170
MONO = 'DejaVu Sans Mono'


def tech_canvas(figsize):
    """Standard light tech canvas with subtle dot grid background."""
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(length=0)
    return fig, ax


def dot_grid(ax, xmin, xmax, ymin, ymax, step=0.25, color=BORDER, size=2):
    xs = np.arange(xmin, xmax + step / 2, step)
    ys = np.arange(ymin, ymax + step / 2, step)
    xv, yv = np.meshgrid(xs, ys)
    ax.scatter(xv.flatten(), yv.flatten(), s=size,
               color=color, zorder=0, alpha=0.7)


def title_block(ax, x, y, title, subtitle=None, ha='left'):
    ax.text(x, y, title, fontsize=15.5, fontweight='bold',
            color=INK, ha=ha, va='bottom')
    if subtitle:
        ax.text(x, y - 0.04, subtitle, fontsize=9.5, color=INK_MUTED,
                family=MONO, ha=ha, va='top')


def card(ax, x, y, w, h, fill=SURFACE, border=BORDER, lw=1.1,
         radius=0.04, shadow=True, accent=None, accent_w=0.06,
         zorder=2):
    if shadow:
        sh = FancyBboxPatch((x + 0.025, y - 0.04), w, h,
                            boxstyle=f'round,pad=0.01,rounding_size={radius}',
                            facecolor='#0F172A', edgecolor='none',
                            alpha=0.05, zorder=zorder - 1)
        ax.add_patch(sh)
    box = FancyBboxPatch((x, y), w, h,
                         boxstyle=f'round,pad=0.01,rounding_size={radius}',
                         facecolor=fill, edgecolor=border, linewidth=lw,
                         zorder=zorder)
    ax.add_patch(box)
    if accent:
        accent_rect = FancyBboxPatch((x, y), accent_w, h,
                                     boxstyle=f'round,pad=0.01,rounding_size={radius}',
                                     facecolor=accent, edgecolor='none',
                                     zorder=zorder + 1)
        ax.add_patch(accent_rect)
        clip = Rectangle((x + accent_w * 0.55, y), w - accent_w * 0.55, h,
                         facecolor=fill, edgecolor='none', zorder=zorder + 2)
        ax.add_patch(clip)


def chip(ax, x, y, text, fill=TINT['slate'], color=INK_SOFT,
         pad_w=0.06, pad_h=0.04, fontsize=8.5, mono=True, weight='bold',
         border=None, zorder=5):
    family = MONO if mono else None
    txt = ax.text(x, y, text, fontsize=fontsize, color=color,
                  family=family, fontweight=weight,
                  ha='center', va='center', zorder=zorder + 1)
    # Manual sized chip approximation using bbox
    txt.set_bbox(dict(boxstyle=f'round,pad=0.35',
                      facecolor=fill, edgecolor=border if border else 'none',
                      linewidth=0.8))
    return txt


## Figura 1 - Matrice di prioritizzazione dei casi d'uso

Quadranti Value × Feasibility con quattro casi d'uso. Caso pilota evidenziato in rose.


In [2]:
def fig1_use_cases():
    fig, ax = tech_canvas((13, 8.5))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('auto')

    # quadrant tints
    quad_tints = {
        (1, 1): TINT['emerald'],
        (0, 1): TINT['amber'],
        (1, 0): TINT['blue'],
        (0, 0): TINT['slate'],
    }
    for (qx, qy), color in quad_tints.items():
        x0 = 0.5 if qx else 0.0
        y0 = 0.5 if qy else 0.0
        ax.add_patch(Rectangle((x0, y0), 0.5, 0.5,
                               facecolor=color, edgecolor='none',
                               alpha=0.45, zorder=0))

    dot_grid(ax, 0, 1, 0, 1, step=0.05, color=BORDER, size=1.0)

    # quadrant labels (top-right corner badges)
    quad_labels = [
        (0.96, 0.96, 'Quick Wins',           ACCENT['emerald'], 'right', 'top'),
        (0.04, 0.96, 'Strategic Initiatives', ACCENT['amber'],   'left',  'top'),
        (0.96, 0.04, 'Fill-ins',             ACCENT['blue'],    'right', 'bottom'),
        (0.04, 0.04, 'Time Sinks',           ACCENT['slate'],   'left',  'bottom'),
    ]
    for x, y, label, color, ha, va in quad_labels:
        ax.text(x, y, label.upper(), fontsize=10, fontweight='bold',
                family=MONO, color=color, ha=ha, va=va, zorder=4)

    # axes
    ax.axhline(0.5, color=BORDER, linewidth=1.4, zorder=1)
    ax.axvline(0.5, color=BORDER, linewidth=1.4, zorder=1)

    use_cases = [
        {'name': 'Pricing dinamico',           'value': 0.90, 'feas': 0.86, 'pilot': True},
        {'name': 'Recommendation',             'value': 0.80, 'feas': 0.66, 'pilot': True},
        {'name': 'Demand forecasting',         'value': 0.62, 'feas': 0.86, 'pilot': False},
        {'name': 'Ottimizz. assortimento',     'value': 0.62, 'feas': 0.40, 'pilot': False},
    ]

    label_offsets = {
        'Pricing dinamico':       (-0.025, -0.018, 'right', 'top'),
        'Recommendation':         ( 0.025, -0.018, 'left',  'top'),
        'Demand forecasting':     (-0.025,  0.020, 'right', 'bottom'),
        'Ottimizz. assortimento': ( 0.025,  0.020, 'left',  'bottom'),
    }

    for uc in use_cases:
        color = ACCENT['rose'] if uc['pilot'] else ACCENT['slate']
        # outer halo
        ax.scatter(uc['feas'], uc['value'], s=700 if uc['pilot'] else 360,
                   color=color, alpha=0.18, zorder=2)
        ax.scatter(uc['feas'], uc['value'], s=240 if uc['pilot'] else 140,
                   color=color, edgecolor='white', linewidth=2.4, zorder=3)
        dx, dy, ha, va = label_offsets[uc['name']]
        weight = 'bold' if uc['pilot'] else 'normal'
        ax.text(uc['feas'] + dx, uc['value'] + dy, uc['name'],
                fontsize=10.5, fontweight=weight, color=INK,
                ha=ha, va=va, zorder=4,
                bbox=dict(boxstyle='round,pad=0.35',
                          facecolor=SURFACE, edgecolor=BORDER,
                          linewidth=0.9))

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('FATTIBILITÀ DI GOVERNANCE  →', fontsize=10, family=MONO,
                  color=INK_SOFT, labelpad=12)
    ax.set_ylabel('VALORE DI BUSINESS  →', fontsize=10, family=MONO,
                  color=INK_SOFT, labelpad=12)

    legend = [
        Line2D([0], [0], marker='o', color='none', label='Caso d\'uso pilota',
               markerfacecolor=ACCENT['rose'], markeredgecolor='white',
               markersize=12, markeredgewidth=1.5),
        Line2D([0], [0], marker='o', color='none', label='Caso d\'uso secondario',
               markerfacecolor=ACCENT['slate'], markeredgecolor='white',
               markersize=10, markeredgewidth=1.5),
    ]
    ax.legend(handles=legend, loc='lower left',
              bbox_to_anchor=(0.0, -0.16), ncol=2,
              fontsize=10, frameon=False)

    title_block(ax, 0, 1.08, 'Prioritizzazione dei casi d\'uso',
                subtitle='MATRICE  ·  VALUE  ×  FEASIBILITY')
    plt.tight_layout()
    plt.savefig('01_use_case_prioritization.png', dpi=DPI,
                bbox_inches='tight', facecolor=BG)
    plt.close(fig)


# === Figure 2 - Logical architecture =====================================

fig1_use_cases()


## Figura 2 - Architettura logica

Otto domini funzionali in quattro layer: presentazione, orchestrazione, core, integrazione.


In [3]:
def fig2_architecture():
    fig, ax = tech_canvas((14.5, 9.5))
    ax.set_xlim(0, 14.5)
    ax.set_ylim(0, 9.6)

    dot_grid(ax, 0, 14.5, 0, 9.6, step=0.4, color=BORDER, size=1.0)

    layers = [
        ('PRESENTAZIONE',  7.6, ACCENT['blue']),
        ('ORCHESTRAZIONE', 5.7, ACCENT['amber']),
        ('CORE',           3.5, ACCENT['emerald']),
        ('INTEGRAZIONE',   1.2, ACCENT['rose']),
    ]
    for name, y, color in layers:
        ax.text(0.6, y + 0.6, name, fontsize=9.5, family=MONO,
                color=color, fontweight='bold', ha='left', va='bottom')
        ax.plot([0.6, 13.9], [y + 0.55, y + 0.55],
                color=BORDER, linewidth=0.8, zorder=1)

    def domain_card(x, y, w, h, idx, name, accent_color):
        card(ax, x, y, w, h, accent=accent_color, accent_w=0.18)
        # ID chip top-right
        ax.text(x + w - 0.25, y + h - 0.25, f'D{idx}',
                fontsize=9, family=MONO, fontweight='bold',
                color=accent_color, ha='right', va='top',
                bbox=dict(boxstyle='round,pad=0.3',
                          facecolor=BG, edgecolor=BORDER, linewidth=0.8),
                zorder=5)
        ax.text(x + 0.45, y + h / 2, name, fontsize=10.5, color=INK,
                fontweight='bold', ha='left', va='center', zorder=5)

    # Presentation layer
    domain_card(1.8, 7.6, 5.4, 1.3, 6, 'Reporting & Dashboard',  ACCENT['blue'])
    domain_card(7.8, 7.6, 5.4, 1.3, 7, 'Access Control & Governance', ACCENT['blue'])
    # Orchestration
    domain_card(4.8, 5.7, 5.4, 1.3, 3, 'Lifecycle Management',   ACCENT['amber'])
    # Core
    domain_card(1.0, 3.5, 2.95, 1.3, 1, 'Catalogo & metadati',   ACCENT['emerald'])
    domain_card(4.25, 3.5, 2.95, 1.3, 2, 'Policy & Rule Management', ACCENT['emerald'])
    domain_card(7.5, 3.5, 2.95, 1.3, 4, 'Audit & Compliance',    ACCENT['emerald'])
    domain_card(10.75, 3.5, 2.95, 1.3, 5, 'Monitoring & Alerting', ACCENT['emerald'])
    # Integration
    domain_card(1.8, 1.2, 11.4, 1.3, 8, 'Integrazione pipeline / sistemi cliente / ticketing', ACCENT['rose'])

    def connect(x1, y1, x2, y2):
        a = FancyArrowPatch((x1, y1), (x2, y2),
                            arrowstyle='-', color=INK_MUTED,
                            linewidth=1.1, alpha=0.85, zorder=2)
        ax.add_patch(a)

    # presentation <-> orchestration
    connect(4.5, 7.6, 6.8, 7.0)
    connect(10.5, 7.6, 8.2, 7.0)
    # orchestration <-> core (4 children)
    connect(6.0, 5.7, 2.5, 4.8)
    connect(7.0, 5.7, 5.7, 4.8)
    connect(8.0, 5.7, 9.0, 4.8)
    connect(9.0, 5.7, 12.2, 4.8)
    # core <-> integration
    connect(2.5, 3.5, 4.5, 2.5)
    connect(5.7, 3.5, 6.5, 2.5)
    connect(9.0, 3.5, 8.5, 2.5)
    connect(12.2, 3.5, 10.5, 2.5)

    ax.set_xticks([])
    ax.set_yticks([])
    title_block(ax, 0.5, 9.5, 'Architettura logica della piattaforma',
                subtitle='OTTO DOMINI FUNZIONALI  ·  LAYERED VIEW')
    plt.tight_layout()
    plt.savefig('02_logical_architecture.png', dpi=DPI,
                bbox_inches='tight', facecolor=BG)
    plt.close(fig)


# === Figure 3 - Approval workflow ========================================

fig2_architecture()


## Figura 3 - Workflow di approvazione del ciclo di vita modello

Stati canonici, gate di approvazione e rollback path verso lo stato di Ritirato.


In [4]:
def fig3_workflow():
    fig, ax = tech_canvas((14.5, 6.8))
    ax.set_xlim(0, 14.5)
    ax.set_ylim(0, 6.5)

    dot_grid(ax, 0, 14.5, 0, 6.5, step=0.4, color=BORDER, size=1.0)

    def state(x, y, w, h, label, accent):
        card(ax, x, y, w, h, accent=accent, accent_w=0.16, radius=0.05)
        ax.text(x + 0.35, y + h - 0.28, 'STATO',
                fontsize=7.5, family=MONO, color=INK_MUTED,
                fontweight='bold', ha='left', va='top', zorder=5)
        ax.text(x + w / 2 + 0.08, y + h / 2 - 0.05, label,
                fontsize=11.5, fontweight='bold', color=INK,
                ha='center', va='center', zorder=5)

    def gate(x, y, top, bottom):
        # rounded square (hexagon-like) rotated 45
        size = 0.55
        diamond = Polygon([(x, y + size), (x + size, y),
                           (x, y - size), (x - size, y)],
                          facecolor=SURFACE, edgecolor=ACCENT['amber'],
                          linewidth=2.2, zorder=4)
        ax.add_patch(diamond)
        # subtle drop shadow ring
        ax.add_patch(Polygon([(x, y + size - 0.04), (x + size - 0.04, y),
                              (x, y - size + 0.04), (x - size + 0.04, y)],
                             facecolor=TINT['amber'], edgecolor='none',
                             zorder=3))
        ax.text(x, y + 0.13, top, fontsize=8.8, fontweight='bold',
                ha='center', va='center', color=INK, zorder=5)
        ax.text(x, y - 0.13, bottom, fontsize=7.8, family=MONO,
                ha='center', va='center', color=ACCENT['amber'], zorder=5)

    def arrow(x1, y1, x2, y2, color=INK_MUTED, style='-', dashed=False, label=None):
        a = FancyArrowPatch((x1, y1), (x2, y2),
                            arrowstyle='->', mutation_scale=18,
                            color=color, linewidth=1.6,
                            linestyle='--' if dashed else style, zorder=3)
        ax.add_patch(a)

    y_main = 4.0
    state(0.4, y_main - 0.55, 1.8, 1.1, 'Sviluppo',    ACCENT['slate'])
    state(3.8, y_main - 0.55, 1.8, 1.1, 'Test',        ACCENT['cyan'])
    state(7.4, y_main - 0.55, 1.8, 1.1, 'Staging',     ACCENT['violet'])
    state(11.4, y_main - 0.55, 2.1, 1.1, 'Produzione', ACCENT['emerald'])

    state(11.5, 1.1, 1.9, 1.1, 'Deprecato', ACCENT['amber'])
    state(8.2, 1.1, 1.9, 1.1,  'Ritirato',  ACCENT['rose'])

    gate(2.9, y_main, 'Test', 'auto')
    gate(6.5, y_main, 'Technical', 'sign-off')
    gate(10.3, y_main, 'Compliance', 'sign-off')

    arrow(2.2, y_main, 2.4, y_main)
    arrow(3.4, y_main, 3.8, y_main)
    arrow(5.6, y_main, 6.0, y_main)
    arrow(7.0, y_main, 7.4, y_main)
    arrow(9.2, y_main, 9.8, y_main)
    arrow(10.8, y_main, 11.4, y_main)
    # produzione -> deprecato
    arrow(12.45, y_main - 0.55, 12.45, 2.2)
    # deprecato -> ritirato
    arrow(11.5, 1.65, 10.1, 1.65)
    # rollback produzione -> ritirato
    rollback = FancyArrowPatch((11.55, y_main - 0.4), (9.15, 2.22),
                               arrowstyle='->', mutation_scale=18,
                               color=ACCENT['rose'], linewidth=1.7,
                               linestyle=(0, (5, 3)),
                               connectionstyle='arc3,rad=-0.28',
                               zorder=3)
    ax.add_patch(rollback)
    ax.text(10.4, 3.05, 'rollback / incidente',
            fontsize=9, family=MONO, color=ACCENT['rose'],
            fontweight='bold', ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.35',
                      facecolor=BG, edgecolor=ACCENT['rose'], linewidth=1.0),
            zorder=5)

    # bottom legend
    chip_legend = [
        ('STATO ATTIVO',     ACCENT['emerald']),
        ('GATE DI APPROV.',  ACCENT['amber']),
        ('FINE VITA',        ACCENT['rose']),
    ]
    for i, (txt, color) in enumerate(chip_legend):
        x = 0.6 + i * 3.0
        ax.add_patch(Circle((x, 0.35), 0.09, color=color, zorder=4))
        ax.text(x + 0.2, 0.35, txt, fontsize=9, family=MONO,
                color=INK_SOFT, fontweight='bold', va='center', zorder=4)

    ax.set_xticks([])
    ax.set_yticks([])
    title_block(ax, 0.4, 6.35, 'Workflow di approvazione e ciclo di vita',
                subtitle='STATE MACHINE  ·  SVILUPPO → PRODUZIONE → END OF LIFE')
    plt.tight_layout()
    plt.savefig('03_approval_workflow.png', dpi=DPI,
                bbox_inches='tight', facecolor=BG)
    plt.close(fig)


# === Figure 4 - KPI tree =================================================

fig3_workflow()


## Figura 4 - Albero dei KPI

Quattro categorie (business, operativi, governance, qualità modello) con i rispettivi KPI.


In [5]:
def fig4_kpi_tree():
    fig, ax = tech_canvas((15.5, 9.5))
    ax.set_xlim(0, 15.5)
    ax.set_ylim(0, 9.5)
    dot_grid(ax, 0, 15.5, 0, 9.5, step=0.4, color=BORDER, size=1.0)

    # root
    root_w, root_h = 5.5, 0.9
    root_x = 7.75 - root_w / 2
    root_y = 8.0
    card(ax, root_x, root_y, root_w, root_h,
         fill=ACCENT['blue'], border=ACCENT['blue'], accent=None,
         radius=0.06, shadow=True)
    ax.text(7.75, root_y + root_h / 2, 'Successo della piattaforma di AI Governance',
            fontsize=12, fontweight='bold', color='white',
            ha='center', va='center', zorder=5)
    ax.text(7.75, root_y + root_h + 0.05, 'NORTH STAR',
            fontsize=8.5, family=MONO, color=INK_MUTED,
            ha='center', va='bottom', zorder=5)

    categories = [
        (2.5,  'Business',                 ACCENT['emerald']),
        (6.25, 'Operativi',                ACCENT['cyan']),
        (10.0, 'Governance & Compliance',  ACCENT['amber']),
        (13.5, 'Qualità modello',          ACCENT['rose']),
    ]

    # connectors root -> categories
    for cx, _, _ in categories:
        ax.plot([7.75, cx], [root_y, 7.2],
                color=INK_MUTED, linewidth=1.1, alpha=0.75, zorder=1)

    cat_y = 6.6
    cat_w, cat_h = 3.0, 0.7
    for cx, label, color in categories:
        card(ax, cx - cat_w / 2, cat_y, cat_w, cat_h, accent=color, accent_w=0.12,
             radius=0.05)
        ax.text(cx + 0.05, cat_y + cat_h / 2, label, fontsize=10.5,
                fontweight='bold', color=INK, ha='center', va='center', zorder=5)

    kpis = {
        2.5:  ['Revenue uplift', 'Conversion lift', 'Perdite evitate', 'Margine offerta governata'],
        6.25: ['Time-to-deploy', 'Modelli attivi/cliente', 'Detection drift time',
               'Remediation time', 'Tasso adozione team'],
        10.0: ['% modelli compliant', '% decisioni auditable', 'Non-conformità rilevate',
               'Remediation non-conf.', 'Audit gestiti'],
        13.5: ['Metrica tecnica primaria', 'Fairness aggregato', 'Privacy adherence',
               'Stale model rate'],
    }
    color_for = {2.5: ACCENT['emerald'], 6.25: ACCENT['cyan'],
                 10.0: ACCENT['amber'], 13.5: ACCENT['rose']}

    for cx, labels in kpis.items():
        n = len(labels)
        ys = np.linspace(5.7, 0.8, n)
        accent_c = color_for[cx]
        # vertical stem
        ax.plot([cx, cx], [cat_y, ys[-1] + 0.27],
                color=BORDER, linewidth=1.2, zorder=1)
        for label, y in zip(labels, ys):
            ax.plot([cx, cx], [y + 0.27, y + 0.27], color=accent_c,
                    linewidth=2, zorder=1)
            # short horizontal tick
            ax.plot([cx - 0.05, cx + 0.05], [y, y], color=accent_c,
                    linewidth=1.5, zorder=2)
            card(ax, cx - 1.4, y - 0.22, 2.8, 0.44,
                 accent=accent_c, accent_w=0.08, radius=0.04, shadow=False)
            ax.text(cx + 0.05, y, label, fontsize=9.3, color=INK,
                    ha='center', va='center', zorder=5)

    ax.set_xticks([])
    ax.set_yticks([])
    title_block(ax, 0.4, 9.25, 'Albero dei KPI della piattaforma',
                subtitle='QUATTRO CATEGORIE  ·  METRICHE TATTICHE')
    plt.tight_layout()
    plt.savefig('04_kpi_tree.png', dpi=DPI,
                bbox_inches='tight', facecolor=BG)
    plt.close(fig)


# === Figure 5 - Roadmap Gantt ============================================

fig4_kpi_tree()


## Figura 5 - Roadmap Gantt

PoC (settimane), MVP (mesi), prodotto scalato (mesi), con milestone diamanti.


In [6]:
def fig5_gantt():
    fig, ax = tech_canvas((15.5, 9))
    MONTHS_TOTAL = 18

    tasks = [
        {'name': 'Raccolta requisiti pilota',         'start': 0.0,  'end': 0.5,  'phase': 'poc'},
        {'name': 'Design flusso approvazione',        'start': 0.4,  'end': 1.2,  'phase': 'poc'},
        {'name': 'Mock-up dashboard',                 'start': 0.7,  'end': 1.6,  'phase': 'poc'},
        {'name': 'Simulazioni monitoring',            'start': 1.2,  'end': 2.0,  'phase': 'poc'},
        {'name': 'Build catalogo + lifecycle',        'start': 2.0,  'end': 5.0,  'phase': 'mvp'},
        {'name': 'Workflow di approvazione',          'start': 3.0,  'end': 6.0,  'phase': 'mvp'},
        {'name': 'Monitoring & alerting',             'start': 4.0,  'end': 7.0,  'phase': 'mvp'},
        {'name': 'Audit & reporting',                 'start': 5.0,  'end': 8.0,  'phase': 'mvp'},
        {'name': 'Migrazione altri clienti (ondate)', 'start': 8.0,  'end': 14.0, 'phase': 'scale'},
        {'name': 'Estensione famiglie modello',       'start': 9.0,  'end': 15.0, 'phase': 'scale'},
        {'name': 'Framework policy riutilizzabile',   'start': 10.0, 'end': 14.0, 'phase': 'scale'},
        {'name': 'Offerta AI-as-a-Service governata', 'start': 11.0, 'end': 18.0, 'phase': 'scale'},
    ]
    color_for = {'poc': ACCENT['blue'], 'mvp': ACCENT['emerald'], 'scale': ACCENT['amber']}

    ax.set_xlim(-1.0, MONTHS_TOTAL + 0.3)
    ax.set_ylim(-3.0, len(tasks) + 4.6)

    # phase bands
    for x0, x1, phase, label in [
        (0, 2, 'poc', 'POC'),
        (2, 8, 'mvp', 'MVP'),
        (8, 18, 'scale', 'PRODOTTO SCALATO'),
    ]:
        ax.axvspan(x0, x1, facecolor=color_for[phase], alpha=0.07, zorder=0)
        ax.plot([x0, x0], [-2.4, len(tasks) + 2.6], color=color_for[phase],
                linewidth=1.3, alpha=0.7, zorder=1)

    # phase header chips
    headers = [(1.0, 'POC', 'granularità: settimane', ACCENT['blue']),
               (5.0, 'MVP', 'granularità: mesi',     ACCENT['emerald']),
               (13.0,'PRODOTTO SCALATO', 'granularità: mesi', ACCENT['amber'])]
    for x, h, sub, color in headers:
        ax.text(x, len(tasks) + 2.1, h, fontsize=12, fontweight='bold',
                family=MONO, color=color, ha='center', va='center', zorder=4,
                bbox=dict(boxstyle='round,pad=0.45', facecolor=SURFACE,
                          edgecolor=color, linewidth=1.4))
        ax.text(x, len(tasks) + 1.35, sub, fontsize=8.5, family=MONO,
                color=INK_MUTED, ha='center', va='center', zorder=4)

    # bars
    LABEL_MIN_WIDTH = 1.7
    for i, t in enumerate(tasks):
        y = len(tasks) - i - 1
        duration = t['end'] - t['start']
        color = color_for[t['phase']]
        # bar shadow
        ax.add_patch(FancyBboxPatch((t['start'] + 0.04, y - 0.30), duration, 0.55,
                                    boxstyle='round,pad=0,rounding_size=0.12',
                                    facecolor='#0F172A', alpha=0.08,
                                    edgecolor='none', zorder=2))
        # bar
        ax.add_patch(FancyBboxPatch((t['start'], y - 0.25), duration, 0.55,
                                    boxstyle='round,pad=0,rounding_size=0.12',
                                    facecolor=color, alpha=0.95,
                                    edgecolor='white', linewidth=1.5, zorder=3))
        if duration >= LABEL_MIN_WIDTH:
            ax.text(t['start'] + 0.18, y + 0.02, t['name'],
                    va='center', ha='left', fontsize=9.5,
                    color='white', fontweight='bold', zorder=4)
        else:
            ax.text(t['end'] + 0.2, y + 0.02, t['name'],
                    va='center', ha='left', fontsize=9.5,
                    color=color, fontweight='bold', zorder=4)

    # milestones
    milestones = [
        (2,  'Sign-off PoC',         ACCENT['blue']),
        (8,  'MVP rilasciato',       ACCENT['emerald']),
        (14, 'Adozione 50%+',        ACCENT['amber']),
        (18, 'Target KPI raggiunti', ACCENT['rose']),
    ]
    for m_x, m_label, m_color in milestones:
        ax.plot(m_x, -0.8, marker='D', markersize=15,
                color=m_color, markeredgecolor='white',
                markeredgewidth=2.2, zorder=5)
        ax.text(m_x, -1.7, m_label, ha='center', va='center', fontsize=9.5,
                color=m_color, fontweight='bold', zorder=5,
                bbox=dict(boxstyle='round,pad=0.45', facecolor=SURFACE,
                          edgecolor=m_color, linewidth=1.2))

    # x-axis as month chips
    ax.set_yticks([])
    ax.set_xticks([])
    for i in range(0, MONTHS_TOTAL + 1):
        ax.text(i, -2.5, f'M{i}', fontsize=8.5, family=MONO,
                color=INK_MUTED, ha='center', va='center')
    ax.axhline(-2.15, color=BORDER, linewidth=1.0, zorder=1)
    ax.text(0, -2.85, 'TIMELINE (mesi dalla decisione di avvio)',
            fontsize=9, family=MONO, color=INK_SOFT,
            fontweight='bold', ha='left', va='top')

    title_block(ax, 0, len(tasks) + 4.3, 'Roadmap di sviluppo',
                subtitle='POC  ·  MVP  ·  PRODOTTO SCALATO')
    plt.tight_layout()
    plt.savefig('05_roadmap_gantt.png', dpi=DPI,
                bbox_inches='tight', facecolor=BG)
    plt.close(fig)


# === Figure 6 - Risk heatmap =============================================

fig5_gantt()


## Figura 6 - Risk heatmap

Probabilità × impatto, ogni rischio rappresentato come chip monospace dentro la cella.


In [7]:
def fig6_risk_heatmap():
    fig, ax = tech_canvas((11.5, 10.5))
    ax.set_xlim(-0.5, 4.05)
    ax.set_ylim(-1.05, 5.0)
    ax.set_aspect('auto')

    # severity tiers: score = p + i (2..8). We use cell-specific colour ramps.
    grid_colors = [
        [TINT['emerald'], TINT['emerald'], TINT['amber'], TINT['amber']],
        [TINT['emerald'], TINT['amber'],   TINT['amber'], TINT['rose']],
        [TINT['amber'],   TINT['amber'],   TINT['rose'],  TINT['rose']],
        [TINT['amber'],   TINT['rose'],    TINT['rose'],  '#FCA5A5'],
    ]
    border_colors = [
        [ACCENT['emerald'], ACCENT['emerald'], ACCENT['amber'], ACCENT['amber']],
        [ACCENT['emerald'], ACCENT['amber'],   ACCENT['amber'], ACCENT['rose']],
        [ACCENT['amber'],   ACCENT['amber'],   ACCENT['rose'],  ACCENT['rose']],
        [ACCENT['amber'],   ACCENT['rose'],    ACCENT['rose'],  ACCENT['rose']],
    ]

    for i in range(4):
        for j in range(4):
            ax.add_patch(FancyBboxPatch((j + 0.03, i + 0.03), 0.94, 0.94,
                                        boxstyle='round,pad=0,rounding_size=0.06',
                                        facecolor=grid_colors[i][j],
                                        edgecolor=border_colors[i][j],
                                        linewidth=1.4, alpha=0.95, zorder=2))

    risks = [
        {'id': 'RT-01', 'p': 3, 'i': 3},
        {'id': 'RT-02', 'p': 2, 'i': 3},
        {'id': 'RT-03', 'p': 3, 'i': 2},
        {'id': 'RT-04', 'p': 2, 'i': 3},
        {'id': 'RC-01', 'p': 2, 'i': 4},
        {'id': 'RC-02', 'p': 1, 'i': 4},
        {'id': 'RC-03', 'p': 1, 'i': 4},
        {'id': 'RC-04', 'p': 2, 'i': 3},
        {'id': 'RC-05', 'p': 1, 'i': 3},
        {'id': 'RO-01', 'p': 3, 'i': 3},
        {'id': 'RO-02', 'p': 2, 'i': 3},
        {'id': 'RO-03', 'p': 2, 'i': 2},
        {'id': 'RO-04', 'p': 3, 'i': 2},
        {'id': 'RO-05', 'p': 3, 'i': 2},
    ]

    cell_risks = defaultdict(list)
    for r in risks:
        cell_risks[(r['p'], r['i'])].append(r['id'])

    for (p, i), ids in cell_risks.items():
        n = len(ids)
        cell_x = p - 1 + 0.5
        cell_y = i - 1 + 0.5
        # arrange in compact grid inside cell
        cols = 2 if n > 2 else 1
        rows = (n + cols - 1) // cols
        chip_w = 0.36
        chip_h = 0.22
        gap_x = 0.04
        gap_y = 0.05
        total_w = cols * chip_w + (cols - 1) * gap_x
        total_h = rows * chip_h + (rows - 1) * gap_y
        x0 = cell_x - total_w / 2
        y0 = cell_y - total_h / 2
        accent_c = border_colors[i - 1][p - 1]
        for k, rid in enumerate(ids):
            r = k // cols
            c = k % cols
            cx = x0 + c * (chip_w + gap_x)
            cy = y0 + (rows - 1 - r) * (chip_h + gap_y)
            ax.add_patch(FancyBboxPatch((cx, cy), chip_w, chip_h,
                                        boxstyle='round,pad=0,rounding_size=0.06',
                                        facecolor=SURFACE,
                                        edgecolor=accent_c, linewidth=1.0,
                                        zorder=4))
            ax.text(cx + chip_w / 2, cy + chip_h / 2, rid,
                    fontsize=8, family=MONO, fontweight='bold',
                    color=accent_c, ha='center', va='center', zorder=5)

    p_labels = ['Bassa', 'Media-bassa', 'Media-alta', 'Alta']
    i_labels = ['Basso', 'Medio', 'Alto', 'Critico']
    for j, lab in enumerate(p_labels):
        ax.text(j + 0.5, -0.15, lab.upper(), fontsize=9, family=MONO,
                color=INK_SOFT, ha='center', va='top')
    for i, lab in enumerate(i_labels):
        ax.text(-0.12, i + 0.5, lab.upper(), fontsize=9, family=MONO,
                color=INK_SOFT, ha='right', va='center', rotation=0)

    ax.text(2.0, -0.35, 'PROBABILITÀ  →', fontsize=10.5, family=MONO,
            color=INK, fontweight='bold', ha='center', va='top')
    ax.text(-0.42, 2.0, 'IMPATTO  →', fontsize=10.5, family=MONO,
            color=INK, fontweight='bold', ha='center', va='center', rotation=90)

    # severity legend at bottom
    severity_legend = [
        ('Tollerabile', ACCENT['emerald']),
        ('Da gestire',  ACCENT['amber']),
        ('Critico',     ACCENT['rose']),
    ]
    for k, (lab, color) in enumerate(severity_legend):
        x = 0.5 + k * 1.2
        ax.add_patch(FancyBboxPatch((x - 0.13, -0.85), 0.20, 0.16,
                                    boxstyle='round,pad=0,rounding_size=0.06',
                                    facecolor=color, edgecolor='none', zorder=4))
        ax.text(x + 0.13, -0.77, lab.upper(), fontsize=9, family=MONO,
                color=INK_SOFT, fontweight='bold', va='center', ha='left', zorder=4)

    ax.set_xticks([])
    ax.set_yticks([])
    title_block(ax, -0.5, 4.85, 'Heatmap dei rischi di progetto',
                subtitle='PROBABILITÀ  ×  IMPATTO  ·  14 RISCHI MAPPATI')
    plt.tight_layout()
    plt.savefig('06_risk_heatmap.png', dpi=DPI,
                bbox_inches='tight', facecolor=BG)
    plt.close(fig)


# === Figure 7 - Stakeholder map ==========================================

fig6_risk_heatmap()


## Figura 7 - Mappa stakeholder

Influenza × interesse, distinzione cromatica tra ruoli interni ed esterni.


In [8]:
def fig7_stakeholders():
    fig, ax = tech_canvas((15, 11))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    quad = [
        ((0.5, 0.5), TINT['emerald']),
        ((0.0, 0.5), TINT['amber']),
        ((0.5, 0.0), TINT['blue']),
        ((0.0, 0.0), TINT['slate']),
    ]
    for (x0, y0), color in quad:
        ax.add_patch(Rectangle((x0, y0), 0.5, 0.5,
                               facecolor=color, edgecolor='none',
                               alpha=0.4, zorder=0))

    dot_grid(ax, 0, 1, 0, 1, step=0.05, color=BORDER, size=0.9)

    ax.axhline(0.5, color=BORDER, linewidth=1.4, zorder=1)
    ax.axvline(0.5, color=BORDER, linewidth=1.4, zorder=1)

    quad_labels = [
        (0.985, 0.985, 'Manage closely (decisori)',            ACCENT['emerald'], 'right', 'top'),
        (0.015, 0.985, 'Keep informed (non bloccanti)',        ACCENT['amber'],   'left',  'top'),
        (0.985, 0.015, 'Keep satisfied (alta influenza)',      ACCENT['blue'],    'right', 'bottom'),
        (0.015, 0.015, 'Monitor (low touch)',                  ACCENT['slate'],   'left',  'bottom'),
    ]
    for x, y, label, color, ha, va in quad_labels:
        ax.text(x, y, label.upper(), fontsize=9.5, fontweight='bold',
                family=MONO, color=color, ha=ha, va=va, zorder=5)

    stakeholders = [
        {'name': 'Sponsor esecutivo',          'inf': 0.95, 'int': 0.90, 'type': 'i', 'dx': -0.025, 'dy':  0.000, 'ha': 'right', 'va': 'center'},
        {'name': 'Practice lead AI',           'inf': 0.82, 'int': 0.83, 'type': 'i', 'dx': -0.025, 'dy':  0.000, 'ha': 'right', 'va': 'center'},
        {'name': 'Sponsor cliente pilota',     'inf': 0.92, 'int': 0.74, 'type': 'e', 'dx': -0.025, 'dy':  0.000, 'ha': 'right', 'va': 'center'},
        {'name': 'CFO / Controllo gestione',   'inf': 0.88, 'int': 0.60, 'type': 'i', 'dx': -0.025, 'dy':  0.000, 'ha': 'right', 'va': 'center'},
        {'name': 'IT/Security cliente',        'inf': 0.80, 'int': 0.52, 'type': 'e', 'dx': -0.025, 'dy':  0.000, 'ha': 'right', 'va': 'center'},
        {'name': 'Garante privacy',            'inf': 0.92, 'int': 0.22, 'type': 'e', 'dx': -0.025, 'dy':  0.000, 'ha': 'right', 'va': 'center'},
        {'name': 'Compliance officer interno', 'inf': 0.66, 'int': 0.91, 'type': 'i', 'dx':  0.025, 'dy':  0.000, 'ha': 'left',  'va': 'center'},
        {'name': 'Legal counsel interno',      'inf': 0.62, 'int': 0.56, 'type': 'i', 'dx': -0.025, 'dy':  0.000, 'ha': 'right', 'va': 'center'},
        {'name': 'Delivery lead',              'inf': 0.66, 'int': 0.78, 'type': 'i', 'dx':  0.025, 'dy':  0.000, 'ha': 'left',  'va': 'center'},
        {'name': 'Data scientist',             'inf': 0.28, 'int': 0.83, 'type': 'i', 'dx':  0.025, 'dy':  0.000, 'ha': 'left',  'va': 'center'},
        {'name': 'ML engineer',                'inf': 0.32, 'int': 0.70, 'type': 'i', 'dx':  0.025, 'dy':  0.000, 'ha': 'left',  'va': 'center'},
        {'name': 'Compliance officer cliente', 'inf': 0.54, 'int': 0.86, 'type': 'e', 'dx': -0.025, 'dy':  0.000, 'ha': 'right', 'va': 'center'},
        {'name': 'Business owner cliente',     'inf': 0.58, 'int': 0.66, 'type': 'e', 'dx': -0.025, 'dy':  0.000, 'ha': 'right', 'va': 'center'},
    ]

    for s in stakeholders:
        color = ACCENT['blue'] if s['type'] == 'i' else ACCENT['rose']
        ax.scatter(s['inf'], s['int'], s=540, color=color, alpha=0.15, zorder=2)
        ax.scatter(s['inf'], s['int'], s=190, color=color,
                   edgecolor='white', linewidth=2.2, zorder=3)
        ax.text(s['inf'] + s['dx'], s['int'] + s['dy'], s['name'],
                fontsize=9.5, color=INK, ha=s['ha'], va=s['va'], zorder=4,
                bbox=dict(boxstyle='round,pad=0.35', facecolor=SURFACE,
                          edgecolor=BORDER, linewidth=0.8))

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('INFLUENZA  →', fontsize=11, family=MONO,
                  color=INK_SOFT, labelpad=14)
    ax.set_ylabel('INTERESSE  →', fontsize=11, family=MONO,
                  color=INK_SOFT, labelpad=14)

    legend = [
        Line2D([0], [0], marker='o', color='none', label='Stakeholder interno (OmniRetail)',
               markerfacecolor=ACCENT['blue'], markeredgecolor='white',
               markersize=12, markeredgewidth=1.6),
        Line2D([0], [0], marker='o', color='none', label='Stakeholder esterno (cliente / regolatore)',
               markerfacecolor=ACCENT['rose'], markeredgecolor='white',
               markersize=12, markeredgewidth=1.6),
    ]
    ax.legend(handles=legend, loc='upper center',
              bbox_to_anchor=(0.5, -0.09), ncol=2,
              fontsize=11, frameon=False)

    title_block(ax, 0, 1.08, 'Mappa stakeholder',
                subtitle='INFLUENZA  ×  INTERESSE  ·  13 RUOLI MAPPATI')
    plt.tight_layout()
    plt.savefig('07_stakeholder_map.png', dpi=DPI,
                bbox_inches='tight', facecolor=BG)
    plt.close(fig)

fig7_stakeholders()
